# Autonomous Vehicle Obstacle Detection — Exploratory Data Analysis

This notebook provides a comprehensive analysis of the obstacle detection dataset:
- Dataset overview and statistics
- Class distribution visualisation
- Bounding box analysis
- Sample image previews
- Model prediction visualisation

**Run this notebook after preprocessing the dataset with:**
```bash
python src/dataset/preprocess_dataset.py --dataset coco --output-dir data/processed
```

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm

from src.utils.config import load_config

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Setup complete.')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
cfg = load_config(PROJECT_ROOT / 'configs/training_config.yaml')

DATA_DIR  = PROJECT_ROOT / cfg.dataset.get('processed_dir', 'data/processed')
CLASS_NAMES = cfg.dataset.get('class_names', [
    'pedestrian', 'bicycle', 'car', 'motorcycle',
    'bus', 'truck', 'traffic_light', 'stop_sign'
])

print(f'Dataset directory : {DATA_DIR}')
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')

## 1. Dataset Overview

In [ ]:
def count_split_stats(data_dir: Path, splits=('train','val','test')):
    rows = []
    for split in splits:
        img_dir = data_dir / 'images' / split
        lbl_dir = data_dir / 'labels' / split
        if not img_dir.exists():
            continue
        images = list(img_dir.glob('*.*'))
        labels = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
        total_ann = sum(len(f.read_text().strip().splitlines()) for f in labels)
        rows.append({
            'Split': split.capitalize(),
            'Images': len(images),
            'Labels': len(labels),
            'Annotations': total_ann,
            'Avg Ann/Image': round(total_ann / max(1, len(images)), 1)
        })
    return pd.DataFrame(rows)

df_stats = count_split_stats(DATA_DIR)
display(df_stats.style.set_caption('Dataset Split Summary'))

## 2. Class Distribution

In [ ]:
def count_classes(data_dir: Path, split: str, class_names: list):
    counts = {name: 0 for name in class_names}
    lbl_dir = data_dir / 'labels' / split
    if not lbl_dir.exists():
        return counts
    for f in tqdm(list(lbl_dir.glob('*.txt')), desc=f'Counting {split}'):
        for line in f.read_text().strip().splitlines():
            parts = line.split()
            if parts:
                cid = int(parts[0])
                if cid < len(class_names):
                    counts[class_names[cid]] += 1
    return counts

splits_data = {}
for split in ('train', 'val', 'test'):
    splits_data[split] = count_classes(DATA_DIR, split, CLASS_NAMES)

df_dist = pd.DataFrame(splits_data).T

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Stacked bar chart
df_dist.plot(kind='bar', ax=axes[0], colormap='tab10', edgecolor='white')
axes[0].set_title('Class Distribution per Split', fontsize=13)
axes[0].set_xlabel('Split')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(loc='upper right', fontsize=8, ncol=2)

# Total donut chart
total = {k: sum(v[k] for v in splits_data.values()) for k in CLASS_NAMES}
wedge_props = dict(width=0.4)
axes[1].pie(
    total.values(), labels=total.keys(),
    autopct='%1.1f%%', pctdistance=0.75,
    wedgeprops=wedge_props
)
axes[1].set_title('Total Class Distribution', fontsize=13)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'runs/eda_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Bounding Box Analysis

In [ ]:
def parse_bboxes(data_dir: Path, split='train'):
    records = []
    lbl_dir = data_dir / 'labels' / split
    if not lbl_dir.exists():
        return pd.DataFrame()
    for f in tqdm(list(lbl_dir.glob('*.txt'))[:5000], desc='Parsing boxes'):
        for line in f.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                cid = int(parts[0])
                cx, cy, w, h = map(float, parts[1:5])
                records.append({
                    'class_id':   cid,
                    'class_name': CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid),
                    'cx': cx, 'cy': cy, 'w': w, 'h': h,
                    'area': w * h,
                    'aspect_ratio': w / (h + 1e-6)
                })
    return pd.DataFrame(records)

df_boxes = parse_bboxes(DATA_DIR, split='train')
print(f'Total annotations parsed: {len(df_boxes):,}')
display(df_boxes.describe().round(4))

In [ ]:
if not df_boxes.empty:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Width dist
    axes[0,0].hist(df_boxes['w'], bins=60, color='steelblue', edgecolor='white')
    axes[0,0].set_title('Box Width Distribution (normalised)')
    axes[0,0].set_xlabel('Width')

    # Height dist
    axes[0,1].hist(df_boxes['h'], bins=60, color='darkorange', edgecolor='white')
    axes[0,1].set_title('Box Height Distribution (normalised)')
    axes[0,1].set_xlabel('Height')

    # Area dist
    axes[0,2].hist(df_boxes['area'], bins=60, color='seagreen', edgecolor='white')
    axes[0,2].set_title('Box Area Distribution (normalised)')
    axes[0,2].set_xlabel('w × h')

    # Aspect ratio
    clip_ar = df_boxes['aspect_ratio'].clip(0, 5)
    axes[1,0].hist(clip_ar, bins=60, color='mediumpurple', edgecolor='white')
    axes[1,0].set_title('Aspect Ratio Distribution (clipped @ 5)')
    axes[1,0].set_xlabel('w / h')

    # cx/cy scatter
    sample = df_boxes.sample(min(5000, len(df_boxes)), random_state=42)
    axes[1,1].scatter(sample['cx'], sample['cy'], s=2, alpha=0.3, c='royalblue')
    axes[1,1].set_title('Box Centre Distribution')
    axes[1,1].set_xlabel('cx')
    axes[1,1].set_ylabel('cy')
    axes[1,1].set_xlim(0,1)
    axes[1,1].set_ylim(1,0)

    # Per-class median size
    median_w = df_boxes.groupby('class_name')['w'].median().sort_values()
    median_w.plot(kind='barh', ax=axes[1,2], color='steelblue', edgecolor='white')
    axes[1,2].set_title('Median Normalised Width per Class')
    axes[1,2].set_xlabel('Median Width')

    plt.suptitle('Bounding Box Statistics — Train Split', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'runs/eda_bbox_stats.png', bbox_inches='tight', dpi=150)
    plt.show()

## 4. Sample Image Visualisation

In [ ]:
from src.dataset.visualize_dataset import visualize_sample_grid

visualize_sample_grid(
    data_dir=DATA_DIR,
    split='train',
    num_samples=16,
    output_dir=PROJECT_ROOT / 'runs',
    seed=42,
)

## 5. Model Prediction Preview (requires trained model)

In [ ]:
WEIGHTS = PROJECT_ROOT / 'models/weights/best.pt'

if WEIGHTS.exists():
    from ultralytics import YOLO
    from src.utils.helper_functions import generate_color_palette, draw_detections
    import random

    model  = YOLO(str(WEIGHTS))
    colors = generate_color_palette(len(CLASS_NAMES))

    test_imgs = list((DATA_DIR / 'images' / 'test').glob('*.*'))[:4]
    if not test_imgs:
        test_imgs = list((DATA_DIR / 'images' / 'val').glob('*.*'))[:4]

    fig, axes = plt.subplots(1, min(4, len(test_imgs)), figsize=(20, 5))
    if len(test_imgs) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, test_imgs[:4]):
        img = cv2.imread(str(img_path))
        results = model.predict(img, conf=0.35, verbose=False)

        boxes, labels, confs, det_colors = [], [], [], []
        for result in results:
            if result.boxes is None:
                continue
            for box in result.boxes:
                cid = int(box.cls.item())
                boxes.append(box.xyxy[0].tolist())
                labels.append(CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else str(cid))
                confs.append(float(box.conf.item()))
                det_colors.append(colors[cid % len(colors)])

        annotated = draw_detections(img, boxes, labels, confs, det_colors)
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(f'{img_path.name}\n{len(boxes)} detections', fontsize=9)

    plt.suptitle('Model Predictions on Test Images', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print(f'No trained model found at {WEIGHTS}.')
    print('Train the model first: python src/training/train.py')

## 6. Summary Table

In [ ]:
if not df_boxes.empty:
    summary = df_boxes.groupby('class_name').agg(
        count=('class_id', 'count'),
        mean_w=('w', 'mean'),
        mean_h=('h', 'mean'),
        mean_area=('area', 'mean'),
        median_ar=('aspect_ratio', 'median')
    ).round(4)
    summary['pct'] = (summary['count'] / summary['count'].sum() * 100).round(1)
    display(summary.sort_values('count', ascending=False)
                   .style.background_gradient(cmap='Blues', subset=['count'])
                   .set_caption('Per-Class Annotation Statistics'))